In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silence per-trial chatter

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

In [ ]:
data = pd.read_csv(r"C:\Users\ykila\Desktop\iCloud\iCloudDrive\Projects\Active\Code\Python\LLM\Aalto\Dataframe.csv")
data["log_pSat"] = np.log10(data["pSat_Pa"])  # compress 10+ orders of magnitude into a tractable range

In [ ]:
# Generate ECFP4 (Morgan radius=2) fingerprints from SMILES — substructural signal not present in RDKit descriptors
def smiles_to_ecfp(smi, n_bits=2048, radius=2):
    arr = np.zeros(n_bits, dtype=np.uint8)
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fps = np.stack([smiles_to_ecfp(s) for s in data["SMILES"]])
fp_df = pd.DataFrame(fps, columns=[f"fp_{i}" for i in range(fps.shape[1])], index=data.index)
print(f"generated {fps.shape[1]}-bit ECFP4 for {fps.shape[0]} molecules")

In [ ]:
X = fps  # ECFP4 fingerprints are the only feature set across all four versions
y = data["log_pSat"].values
print(f"X shape: {X.shape}  (pure {X.shape[1]}-bit ECFP4)")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )


In [ ]:
# Carve 10 % off training to use as validation. Hyperparameter search and
# XGB early stopping both run against this set so X_test stays unobserved
# until the final ensemble line.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [ ]:
# Optuna search over the regularisation-heavy XGB hyperparameters.
# Each trial trains one XGB on (X_tr, y_tr), early-stops against (X_val, y_val),
# and reports val MAE; TPE sampler walks the space.
SEARCH_TRIALS = 30


def objective(trial):
    params = {
        "n_estimators": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 10.0, log=True),
        "tree_method": "hist",
        "objective": "reg:absoluteerror",
        "early_stopping_rounds": 30,
        "random_state": 42,
    }
    m = XGBRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, m.predict(X_val))


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=SEARCH_TRIALS, show_progress_bar=True)

print(f"\nbest val MAE during search: {study.best_value:.4f}")
print("best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Train K XGBs with the best params and different random_state seeds, average
# their test predictions. This is the XGB analogue of v3's NN seed ensemble:
# each tree-bagging seed explores a slightly different subspace, averaging
# decorrelates residual variance.
final_params = {
    **study.best_params,
    "n_estimators": 5000,
    "tree_method": "hist",
    "objective": "reg:absoluteerror",
    "early_stopping_rounds": 50,
}

SEEDS = [0, 1, 2, 3, 4]
seed_test_preds = []
for s in SEEDS:
    m = XGBRegressor(**final_params, random_state=s)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    val_mae = mean_absolute_error(y_val, m.predict(X_val))
    print(f"seed {s}: val MAE = {val_mae:.4f}  (best_iter = {m.best_iteration})")
    seed_test_preds.append(m.predict(X_test))

xgb_test_preds = np.mean(seed_test_preds, axis=0)
print(f"\n{len(SEEDS)}-seed XGB ensemble test MAE: {mean_absolute_error(y_test, xgb_test_preds):.4f}")